# Module 12 · DataFrame and Series

*BUSN 3030E — Python Programming Skills for Business Analytics I*

**Learning objectives.** After this module you will be able to

- create `pandas` **Series** and **DataFrames** and describe their parts: values, index, and columns;
- select data by label with `.loc` and by position with `.iloc`;
- filter rows with Boolean conditions and add computed columns;
- load a dataset from the web and inspect it with `head`, `shape`, `info`, and `dtypes`;
- avoid the most common pitfall of modifying data in pandas 3 (chained assignment).

> This course uses **pandas 3**. If you find older tutorials online, a few details differ; the notes marked *pandas 3* point them out.

## 1 · Preparation

In Module 11 we analyzed a table stored as a list of dictionaries. It worked, but every question took a loop, every missing value needed special care, and the output was hard to read. **pandas** is Python's standard library for tabular data. It provides two main objects:

- a **Series**: a one-dimensional column of values with labels;
- a **DataFrame**: a two-dimensional table made of Series that share the same row labels.

By convention, pandas is imported as `pd`:

In [1]:
import pandas as pd

### 1.1 Series

A Series can be created from a list:

In [2]:
data = pd.Series([0.25, 0.5, 0.75, 1.0])
data

0    0.25
1    0.50
2    0.75
3    1.00
dtype: float64

A Series has two parts: the **values** and the **index** (the labels on the left). By default the index counts from 0:

In [3]:
data.values

array([0.25, 0.5 , 0.75, 1.  ])

In [4]:
data.index

RangeIndex(start=0, stop=4, step=1)

The index can hold any labels. A Series built from a dictionary uses the keys as the index, in the order they were inserted:

In [5]:
population = pd.Series({
    "California": 39_431_263,
    "Texas": 31_290_831,
    "Florida": 23_372_215,
    "New York": 19_867_248,
    "Pennsylvania": 13_078_751,
})
population

California      39431263
Texas           31290831
Florida         23372215
New York        19867248
Pennsylvania    13078751
dtype: int64

*(State population estimates for 2024, US Census Bureau.)*

### 1.2 Selecting from a Series: `.loc` and `.iloc`

There are two ways to point at an item:

- `.loc[...]` selects by **label**;
- `.iloc[...]` selects by integer **position**, starting at 0.

In [6]:
population.loc["Texas"]

31290831

In [7]:
population.iloc[1]

31290831

Both accept slices. Note the difference at the end of the slice: **label slices include the end label**, whereas position slices exclude the end position, as everywhere else in Python.

In [8]:
population.loc["Texas":"New York"]

Texas       31290831
Florida     23372215
New York    19867248
dtype: int64

In [9]:
population.iloc[1:3]

Texas      31290831
Florida    23372215
dtype: int64

Square brackets alone (`population["Texas"]`) also select by label. In pandas 3, plain brackets with an integer *always* mean a label, never a position, so they fail when the index contains no such label:

In [10]:
population[0]

KeyError: 0

> **Rule of thumb:** use `.loc` for labels and `.iloc` for positions. Your code then says exactly what it means.

### 1.3 Vectorized operations

Arithmetic on a Series applies to every value at once, with no loop needed. Operations between two Series line up their values **by label**, not by position.

In [11]:
area = pd.Series({          # land area in square miles
    "Texas": 261_232,
    "California": 155_779,
    "Florida": 53_625,
    "Pennsylvania": 44_743,
    "New York": 47_126,
})

density = population / area   # matched by state name, despite the different order
density.round(1)

California      253.1
Florida         435.8
New York        421.6
Pennsylvania    292.3
Texas           119.8
dtype: float64

### 1.4 Filtering with Boolean conditions

A comparison produces a Series of `True`/`False` values, a **Boolean mask**:

In [12]:
population > 20_000_000

California       True
Texas            True
Florida          True
New York        False
Pennsylvania    False
dtype: bool

Passing a mask inside square brackets keeps the rows where it is `True`:

In [13]:
population[population > 20_000_000]

California    39431263
Texas         31290831
Florida       23372215
dtype: int64

Combine conditions with `&` (and), `|` (or), and `~` (not), and wrap each condition in parentheses. The words `and`/`or` do not work on Series.

In [14]:
population[(population > 15_000_000) & (density > 300)]

Florida     23372215
New York    19867248
dtype: int64

### 1.5 DataFrames

A DataFrame is a table. One way to build it is from a dictionary of Series, one per column:

In [15]:
states = pd.DataFrame({"population": population, "area": area})
states

,population,area
California,39431263,155779
Florida,23372215,53625
New York,19867248,47126
Pennsylvania,13078751,44743
Texas,31290831,261232


The rows are aligned by label, so each state's area lands in the right row. A DataFrame has an index for the rows *and* an index for the columns:

In [16]:
states.index

Index(['California', 'Florida', 'New York', 'Pennsylvania', 'Texas'], dtype='str')

In [17]:
states.columns

Index(['population', 'area'], dtype='str')

In [18]:
states.shape        # (rows, columns)

(5, 2)

**Selecting columns.** A single column name in square brackets returns a Series; a *list* of names returns a DataFrame.

In [19]:
states["area"]

California      155779
Florida          53625
New York         47126
Pennsylvania     44743
Texas           261232
Name: area, dtype: int64

In [20]:
states[["area"]]

,area
California,155779
Florida,53625
New York,47126
Pennsylvania,44743
Texas,261232


**Adding a column.** Assigning to a new column name computes it for every row:

In [21]:
states["density"] = states["population"] / states["area"]
states

,population,area,density
California,39431263,155779,253.123097
Florida,23372215,53625,435.845501
New York,19867248,47126,421.577219
Pennsylvania,13078751,44743,292.308316
Texas,31290831,261232,119.781769


**Selecting rows and cells.** `.loc[row, column]` and `.iloc[row, column]` work in two dimensions. A colon `:` means "all".

In [22]:
states.loc["Florida"]                     # one row, as a Series

population    2.337222e+07
area          5.362500e+04
density       4.358455e+02
Name: Florida, dtype: float64

In [23]:
states.loc["Florida", "density"]          # one cell

435.84550116550116

In [24]:
states.loc[["Texas", "Florida"], ["population", "area"]]

,population,area
Texas,31290831,261232
Florida,23372215,53625


In [25]:
states.iloc[:2, :2]                       # first two rows and columns

,population,area
California,39431263,155779
Florida,23372215,53625


**Filtering and sorting rows.** Boolean masks work on DataFrames too; `sort_values()` orders the rows by a column:

In [26]:
states[states["density"] > 300].sort_values("density", ascending=False)

,population,area,density
Florida,23372215,53625,435.845501
New York,19867248,47126,421.577219


### 1.6 Changing values: avoid chained assignment

Suppose we want to change Florida's area to 54,000 square miles for a what-if scenario. It is tempting to select the column and then the row:

In [27]:
states["area"]["Florida"] = 54_000     # chained assignment: does NOT work in pandas 3
states.loc["Florida", "area"]

ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  states["area"]["Florida"] = 54_000     # chained assignment: does NOT work in pandas 3


53625

pandas 3 warns that nothing changed. In pandas 3, every selection behaves like a new copy (a rule called **Copy-on-Write**), so the assignment modified a temporary copy of the column. The fix is to select the row and the column in a *single* step with `.loc`:

In [28]:
states.loc["Florida", "area"] = 54_000
states.loc["Florida", "area"]

54000

> **pandas 3:** to modify a DataFrame, always use `df.loc[rows, column] = value` or assign a whole column with `df["column"] = ...`. Never chain two selections before `=`.

### 1.7 Loading data from the web

`pd.read_csv()` reads a CSV file into a DataFrame. It accepts a web address directly; the `sep` parameter sets the separator for files that do not use commas (here a tab, `"\t"`).

In [29]:
DATA_URL = "https://raw.githubusercontent.com/hanisaf/python-programming-skills/main/data/"

gapminder = pd.read_csv(DATA_URL + "gapminder.tsv", sep="\t")
gapminder.head()

,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106


These data come from the [Gapminder Foundation](https://www.gapminder.org/data/): life expectancy, population, and GDP per capita (in inflation-adjusted dollars) for 142 countries, every five years from 1952 to 2007.

`info()` summarizes the structure: the number of rows, and each column's name, count of non-missing values, and data type.

In [30]:
gapminder.info()

<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
dtypes: float64(2), int64(2), str(2)
memory usage: 80.0 KB


> **pandas 3:** text columns now have the dedicated data type `str`. Older versions of pandas showed them as `object`.

## 2 · Warmup

**Exercise 2.1.** Using the `population` Series:

- How many entries does it have?
- What are its values and its index?
- What is the population of Florida?
- Extract the states from Texas to New York.

In [31]:
print(len(population))
print(population.to_list())
print(population.index.to_list())
print(population.loc["Florida"])

5
[39431263, 31290831, 23372215, 19867248, 13078751]
['California', 'Texas', 'Florida', 'New York', 'Pennsylvania']
23372215


In [32]:
population.loc["Texas":"New York"]

Texas       31290831
Florida     23372215
New York    19867248
dtype: int64

`to_list()` converts the values or the index to a plain Python list.

**Exercise 2.2.** Build a DataFrame `df` with the columns `population` and `area`. What are its index and columns? How many cells does it contain?

In [33]:
df = pd.DataFrame({"population": population, "area": area})
rows, cols = df.shape
print("index:  ", df.index.to_list())
print("columns:", df.columns.to_list())
print("cells:  ", rows * cols)

index:   ['California', 'Florida', 'New York', 'Pennsylvania', 'Texas']
columns: ['population', 'area']
cells:   10


`df.size` gives the number of cells directly.

## 3 · Workout

*Exercises adapted from Daniel Chen,* Pandas for Everyone.

Use the `gapminder` DataFrame loaded above.

**Exercise 3.1.** How many rows and columns does the dataset have?

In [34]:
gapminder.shape

(1704, 6)

**Exercise 3.2.** Select the `country` column. Then select `country` and `continent` together.

In [35]:
gapminder["country"].head()

0    Afghanistan
1    Afghanistan
2    Afghanistan
3    Afghanistan
4    Afghanistan
Name: country, dtype: str

In [36]:
gapminder[["country", "continent"]].head()

,country,continent
0,Afghanistan,Asia
1,Afghanistan,Asia
2,Afghanistan,Asia
3,Afghanistan,Asia
4,Afghanistan,Asia


**Exercise 3.3.** Select the first row; then the first five rows; then the last five.

In [37]:
gapminder.iloc[0]

country      Afghanistan
continent           Asia
year                1952
lifeExp           28.801
pop              8425333
gdpPercap     779.445314
Name: 0, dtype: object

In [38]:
gapminder.head()

,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106


In [39]:
gapminder.tail()

,country,continent,year,lifeExp,pop,gdpPercap
1699,Zimbabwe,Africa,1987,62.351,9216418,706.157306
1700,Zimbabwe,Africa,1992,60.377,10704340,693.420786
1701,Zimbabwe,Africa,1997,46.809,11404948,792.449960
1702,Zimbabwe,Africa,2002,39.989,11926563,672.038623
1703,Zimbabwe,Africa,2007,43.487,12311143,469.709298


**Exercise 3.4.** Which five countries had the highest life expectancy in 2007?

In [40]:
latest = gapminder[gapminder["year"] == 2007]
latest.sort_values("lifeExp", ascending=False).head(5)

,country,continent,year,lifeExp,pop,gdpPercap
803,Japan,Asia,2007,82.603,127467972,31656.06806
671,"Hong Kong, China",Asia,2007,82.208,6980412,39724.97867
695,Iceland,Europe,2007,81.757,301931,36180.78919
1487,Switzerland,Europe,2007,81.701,7554661,37506.41907
71,Australia,Oceania,2007,81.235,20434176,34435.36744


**Exercise 3.5.** Show the population and GDP per capita of the United States in 1952 and 2007, and compute the total GDP (population × GDP per capita) in billions.

In [41]:
us = gapminder.loc[(gapminder["country"] == "United States") & gapminder["year"].isin([1952, 2007]),
                   ["year", "pop", "gdpPercap"]]
us["gdp_billions"] = us["pop"] * us["gdpPercap"] / 1e9
us

,year,pop,gdpPercap,gdp_billions
1608,1952,157553000,13990.48208,2204.242423
1619,2007,301139947,42951.65309,12934.458535


`isin()` tests membership in a list, the pandas counterpart of the `in` operator. In half a century, the US economy grew almost sixfold in real terms.

## Summary

| Task | Code |
|---|---|
| Create | `pd.Series(list_or_dict)`, `pd.DataFrame({"col": series, ...})` |
| Inspect | `.head()`, `.tail()`, `.shape`, `.info()`, `.dtypes`, `.index`, `.columns` |
| Select column(s) | `df["col"]`, `df[["col1", "col2"]]` |
| Select by label / position | `df.loc[row, col]` / `df.iloc[i, j]` |
| Filter rows | `df[df["col"] > value]`, combine with `&`, `\|`, `~` |
| Sort | `df.sort_values("col", ascending=False)` |
| Add / change | `df["new"] = ...`, `df.loc[row, "col"] = value` |
| Load | `pd.read_csv(url, sep=...)` |